# 03 — Model Development, Evaluation & Explainability

**Student Performance Prediction System**

This notebook covers Module 3:
1. Classical ML model comparisons (Logistic Regression, Decision Tree, Random Forest, HistGradientBoosting)
2. Stratified 5-fold cross-validation & hyperparameter tuning
3. Bootstrapped 95% confidence intervals on Macro-F1 & Accuracy
4. McNemar's statistical test comparing top 2 models
5. SHAP explainability (Global importance + local waterflow explanations)

In [ ]:
# Make `src` importable no matter where Jupyter was launched from.
import sys, warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

print(f"Project root: {ROOT}")

## 1. Load Preprocessed Data & Inspect Targets

In [ ]:
from src.data.preprocess import load_processed, split_features_target, encode_target
from src.utils.config import load_config

cfg = load_config()
df = load_processed(cfg)
X, y = split_features_target(df, cfg)
y_encoded, class_order = encode_target(y, cfg)

print(f"Features: {X.shape[1]} | Samples: {len(X)}")
print(f"Class order: {class_order}")

## 2. Load Evaluation Metrics from Training Run

All metrics are stored in `reports/artifacts/metrics.json` for full reproducibility.

In [ ]:
from src.utils.config import get_path, load_json
import pandas as pd

metrics = load_json(get_path("metrics_file", cfg))
print("Best selected model:", metrics.get("best_model"))
print("Runner-up model:   ", metrics.get("runner_up_model"))

test_results = metrics.get("test_metrics", {})
summary_rows = []
for name, m in test_results.items():
    summary_rows.append({
        "Model": name,
        "Accuracy": m["accuracy"],
        "Macro-F1": m["f1_macro"],
    })
pd.DataFrame(summary_rows).sort_values(by="Macro-F1", ascending=False)

## 3. Bootstrapped Confidence Intervals (95%)

In [ ]:
bootstrap = metrics.get("bootstrap", {})
for model_name, b in bootstrap.items():
    f1_ci = b["f1_macro"]
    print(f"{model_name:20s}: Macro-F1 = {f1_ci['point_estimate']:.4f} [95% CI: {f1_ci['ci_lower']:.4f} - {f1_ci['ci_upper']:.4f}]")

## 4. McNemar's Test: Statistical Significance of Winner vs Runner-up

In [ ]:
mc = metrics.get("mcnemar", {})
print("McNemar's Test:", mc.get("model_a"), "vs", mc.get("model_b"))
print(f"p-value: {mc.get('p_value'):.4e} (Significant: {mc.get('significant')})")
print("Interpretation:", mc.get("interpretation"))

## 5. SHAP Global Feature Importance

In [ ]:
from IPython.display import Image, display
from src.utils.config import get_path

shap_plot = get_path("figures_dir", cfg) / "12_shap_global_importance.png"
if shap_plot.exists():
    display(Image(str(shap_plot)))